In [ ]:
import sqlite3
import warnings
import pandas as pd
from sklearn.metrics import r2_score, accuracy_score, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [17]:
conn = sqlite3.connect('../data/credit_modelling.db')

In [18]:
data = pd.read_sql('SELECT * FROM credit_risk_load_data', conn)
data.head()

,PROSPECTID,pct_tl_open_L6M,pct_tl_closed_L6M,Tot_TL_closed_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,CC_TL,Home_TL,PL_TL,Secured_TL,...,first_prod_enq2_PL,first_prod_enq2_others,last_prod_enq2_AL,last_prod_enq2_CC,last_prod_enq2_ConsumerLoan,last_prod_enq2_HL,last_prod_enq2_PL,last_prod_enq2_others,GENDER_F,GENDER_M
0,1,0.000,0.0,0,0.000,0,0,0,4,1,...,1,0,0,0,0,0,1,0,0,1
1,2,0.000,0.0,0,0.000,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
2,3,0.125,0.0,0,0.000,1,0,0,0,2,...,0,1,0,0,1,0,0,0,0,1
3,5,0.000,0.0,0,0.000,0,0,0,0,3,...,0,0,1,0,0,0,0,0,0,1
4,6,0.000,0.0,1,0.167,0,0,0,0,6,...,1,0,0,0,1,0,0,0,0,1


In [19]:
y = data['Approved_Flag']
x = data.drop(['Approved_Flag'], axis=1)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

rf_classifier = RandomForestClassifier(n_estimators=200, random_state=42)
rf_classifier.fit(x_train, y_train)
y_pred = rf_classifier.predict(x_test)

In [20]:
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.7666706287887792

In [21]:
precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

In [22]:
for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    

Class p1
Precision: 0.8390129259694477
Recall: 0.7041420118343196
F1 Score: 0.7656836461126005
Class p2
Precision: 0.7951096121416527
Recall: 0.9345887016848364
F1 Score: 0.8592255125284738
Class p3
Precision: 0.4563106796116505
Recall: 0.21283018867924527
F1 Score: 0.2902727740607308
Class p4
Precision: 0.7287968441814595
Recall: 0.718172983479106
F1 Score: 0.7234459128732257


In [23]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

xgb_classifier = xgb.XGBClassifier(objective='ulti:softmax', num_class=4)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)

xgb_classifier.fit(x_train, y_train)
y_pred = xgb_classifier.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
accuracy

0.7755854035421371

In [24]:
precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

In [25]:
for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")

Class p1
Precision: 0.8202959830866807
Recall: 0.7652859960552268
F1 Score: 0.7918367346938775
Class p2
Precision: 0.8235294117647058
Recall: 0.9129831516352824
F1 Score: 0.8659522466629066
Class p3
Precision: 0.4645006016847172
Recall: 0.29132075471698116
F1 Score: 0.3580705009276438
Class p4
Precision: 0.725790987535954
Recall: 0.7356656948493683
F1 Score: 0.7306949806949807


In [26]:
# Decision Trees
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier(max_depth=20, min_samples_split=10)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

dt_model.fit(x_train, y_train)
y_pred = dt_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
accuracy

0.7106858433376917

In [27]:
precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

In [28]:
for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")

Class p1
Precision: 0.7335984095427436
Recall: 0.727810650887574
F1 Score: 0.7306930693069307
Class p2
Precision: 0.8096899224806201
Recall: 0.8281466798810704
F1 Score: 0.8188143067123959
Class p3
Precision: 0.3371900826446281
Recall: 0.30792452830188677
F1 Score: 0.32189349112426036
Class p4
Precision: 0.6316297010607522
Recall: 0.6365403304178814
F1 Score: 0.6340755082284608


## Results 

- Random Forest = 76%
- XGBoost = 77.5%
- Decision Tree = 71%

Accuracy wise XGBoost is better at present

Now we will further finetune it :
1. HP Tuning
2. Feature Engineering -- Scaling, Feature Engg, Graphs

If we can get better results

In [29]:
data['Approved_Flag'].value_counts()

Approved_Flag
P2    25452
P3     6440
P4     5264
P1     4908
Name: count, dtype: int64

By looking at the imbalanced nature of the target variable we get to decide the loss metric.

XGBoost is giving the highest accuracy

So we will pick it and further finetune it.

HyperParameters to consider :
1. Max_depth
2. Min_sample_split
3. Learning Rate = Overfitting reduce
4. alpha
5. n_estimators
6. colsample_bytree

Ways to do it : 
- GridSearchCV
- RandomCV
- Bayesian Theorem

Motive is to decide how fast the algorithm converges.